# Codeforces Round 936 (Div. 2)

- https://codeforces.com/contest/1946

`-` 3솔로 끝났다. C번은 트리와 매개 변수 탐색을 합친 문제인데 처음 보는 유형이었다. 그래서 그런지 제대로 접근한 게 맞는데 의심병이 조금 도져서 해결이 늦어졌다 (+ 이 접근이 맞다고? C번 치곤 어렵지 않나? 응 아니야~)

`-` D번은 비트 연산 문제인데 접근법을 1도 모르겠다 (1까진 아니지만 2시간 올인해도 못풀듯)

## A. Median of an Array (00:09)

`-` 중앙값 $a_{\left\lceil\frac{n}{2}\right\rceil}$부터 시작해 오른쪽으로 연속된 $k$개의 값이 동일하다고 하자. 이들 모두에 $1$을 더해야 중앙값이 바뀌므로 정답은 $k$이다 (연속된 구간의 끝에서부터 값을 바꿔 나간다고 생각하면 쉽다)

In [5]:
def solve_testcase(array):
    n = len(array)
    m = (n + 1) // 2 - 1
    array.sort()
    median = array[m]
    return sum(a == median and i >= m for i, a in enumerate(array))


def solution():
    t = int(input())
    for _ in range(t):
        n = int(input())
        a = list(map(int, input().split()))
        answer = solve_testcase(a)
        print(answer)


solution()

# input
# 1
# 5
# 5 5 5 4 5

 1
 5
 5 5 5 4 5


3


`-` 중앙값이랑 같은 값을 가지는 원소가 많은 경우가 복잡했다. 이게 $a_m$ 이전의 원소는 아무런 영향이 없는데 값이 같으니 묶어서 생각하느라 해답을 찾는 데 시간이 오래 겄렸다 (머릿속에서 정렬 시뮬레이션 돌리는 데 과부하돼서 제대로 판단이 안 됨...)

`-` 이것의 악영향으로 본질을 깨닫지 못해 쓸데없이 경우의 수를 쪼개느라 더 오래 걸린 것도 있다. 예컨대 $a_m$과 $a_{m+1}$이 다르면 정답은 $1$이라던지...

## B. Maximum Sum (00:18)

`-` 배열의 합을 최대화하기 위해 최대 연속 부분합을 구하자 (여기선 공집합도 가능하다). 이는 웰노운 문제로 DP를 사용해 $O(n)$에 계산할 수 있다

`-` 이를 $x$라 하자. 매 연산마다 최대 연속 부분합을 인접하게 추가할 것이다. 그럼 $i$번째 연산을 거친 후 배열의 최대 연속 부분합은 $x \cdot 2^i$이 된다

`-` 기존 배열에서 최대 연속 부분합을 제외한 부분은 변하지 않으므로 $\sum\limits_{i=1}^{n}a_i - x + x\cdot 2^k$이 정답이 된다. 이것이 음수가 될 수도 있으므로 모듈로 연산을 할 때 모듈로를 정답에 먼저 더해준 뒤 나머지를 구하자

In [14]:
def solve_testcase(array, k, mod):
    n = len(array)
    dp = [0] * n
    dp[0] = max(array[0], 0)
    for i in range(1, n):
        dp[i] = max(array[i] + dp[i - 1], array[i])
    x = max(dp)
    total = (sum(array) - x + x * pow(2, k, mod) + mod) % mod
    return total


def solution():
    t = int(input())
    mod = 10**9 + 7
    for _ in range(t):
        n, k = map(int, input().split())
        a = list(map(int, input().split()))
        answer = solve_testcase(a, k, mod)
        print(answer)


solution()

# input
# 1
# 5 1
# 4 -2 8 -12 9

 1
 5 1
 4 -2 8 -12 9


17


## C. Tree Cutting (01:15)

`-` 트리와 매개 변수 탐색을 합친 문제이다 (처음 보는 유형임)

`-` 리프 노드부터 시작해 컴포넌트 크기가 $x$ 이상이 되도록 자신과 부모 사이의 간선을 그리디하게 자르자. 컴포넌트가 $k + 1$개 이상이면 $x$를 키우고 아니라면 줄이면 되며 최적의 $x$는 이분 탐색으로 $O(\log N)$에 찾을 수 있다 (간선을 $k$개 제거하면 컴포넌트는 $k+1$개이다)

`-` 전체 알고리즘의 시간 복잡도는 $O(N\log N)$이다

In [13]:
def dfs(tree, root):
    parents = [0] * len(tree)
    dfs_order = []
    stack = [root]
    while stack:
        node = stack.pop()
        dfs_order.append(node)
        for child in tree[node]:
            if child == parents[node]:
                continue
            parents[child] = node
            stack.append(child)
    return parents, dfs_order


def compute_subtree_sizes(tree, parents, dfs_order):
    subtree_sizes = [1] * len(tree)
    for node in reversed(dfs_order):
        subtree_sizes[parents[node]] += subtree_sizes[node]
    return subtree_sizes


def binary_search(tree, parents, subtree_sizes, dfs_order, k):
    low, high = 1, len(tree) // 2
    while low <= high:
        mid = (low + high) // 2
        n_components = cut(parents, subtree_sizes, dfs_order, mid)
        if n_components >= k + 1:
            low = mid + 1
        else:
            high = mid - 1
    return high


def cut(parents, subtree_sizes, dfs_order, x):
    dp = subtree_sizes[:]
    n_components = 0
    for node in reversed(dfs_order):
        dp_node = dp[node]
        dp[parents[node]] -= subtree_sizes[node] - dp_node
        if dp_node >= x:
            dp[parents[node]] -= dp_node
            n_components += 1
    return n_components


def solve_testcase(tree, root, k):
    parents, dfs_order = dfs(tree, root)
    subtree_sizes = compute_subtree_sizes(tree, parents, dfs_order)
    answer = binary_search(tree, parents, subtree_sizes, dfs_order, k)
    return answer


def solution():
    root = 1
    t = int(input())
    for _ in range(t):
        n, k = map(int, input().split())
        tree = [[] for _ in range(n + 1)]
        for _ in range(n - 1):
            v, u = map(int, input().split())
            tree[u].append(v)
            tree[v].append(u)
        answer = solve_testcase(tree, root, k)
        print(answer)


solution()

# input
# 1
# 5 1
# 1 2
# 1 3
# 3 4
# 3 5

 1
 5 1
 1 2
 1 3
 3 4
 3 5


2


`-` TLE가 발생하지 않도록 비재귀로 구현해야 하는 게 가장 어려운 부분이다